# Regression Dataset 실습

이 노트북은 `RegressionDataset`의 이미지 정규화, 레이블 연속값 변환, `Dataloader` 배치 생성 동작을 직접 실행하고 검증하는 실습 자료이다.
이전 노트북의 변수나 실행 결과를 사용하지 않으며, 이 노트북 단독으로 완전히 실행할 수 있다.

**실행 환경**: `conda run -n numpy_py311` (GPU 불필요)

**데이터셋 경로**: `/mnt/d/datasets/mnist/` (train/test `.gz` 4개 파일 필요)

**목표**
- `to_regression`이 정수 레이블을 `[0, 1]` 범위 연속값으로 변환하는 과정을 확인한다.
- `RegressionDataset`이 이미지와 target을 eager하게 변환하여 `__getitem__`에서 인덱싱만 수행함을 확인한다.
- digit별 target 분포 시각화를 통해 정규화 결과를 직관적으로 파악한다.
- `Dataloader`로 배치를 생성하고 학습 루프에서 사용하는 방식을 시연한다.

## 0. 환경 설정

In [ ]:
# sys.path setup -- excluded from jupyter book build
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"project_root={PROJECT_ROOT}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from src.data.mnist import load_images, load_labels
from src.data import transforms as T
from src.data.datasets import RegressionDataset
from src.data.dataloader import Dataloader

## 1. 개요

`RegressionDataset`은 MNIST 숫자 값 자체를 연속값으로 예측하는 regression task용 Dataset이다.
정수 레이블을 `label / 9.0`으로 변환하여 0→0.0, 9→1.0 사이의 연속값으로 매핑한다.
이미지 변환 방식은 `MulticlassDataset`과 동일하며(`normalize + to_flat`), target 형태만 다르다.

각 변환 함수의 역할은 다음과 같다.

| 함수 | 입력 | 출력 | 역할 |
|---|---|---|---|
| `normalize` | `(N, 28, 28) uint8` | `(N, 28, 28) float32` | `/255.0` — 픽셀 값을 `[0, 1]`로 정규화 |
| `to_flat` | `(N, 28, 28) float32` | `(N, 784) float32` | `reshape` — MLP 입력 형태로 펼침 |
| `to_regression` | `(N,) uint8` | `(N, 1) float32` | `/9.0` — 레이블을 `[0, 1]` 연속값으로 정규화 |

예측 결과를 원래 스케일로 복원하려면 9를 곱하고 반올림·클리핑을 적용한다.

## 2. to_regression 변환 확인

`load_labels`가 반환한 원본 `uint8` 레이블 배열에 `to_regression`을 적용한다.
레이블 0~9가 어떤 연속값으로 변환되는지 확인한다.

In [ ]:
labels_raw = load_labels("train")         # (60000,) uint8
labels_reg = T.to_regression(labels_raw)  # (60000, 1) float32

print(f"raw          : shape={labels_raw.shape}, dtype={labels_raw.dtype}")
print(f"to_regression: shape={labels_reg.shape}, dtype={labels_reg.dtype}")
print(f"  min={labels_reg.min():.4f}, max={labels_reg.max():.4f}")
print()
print("레이블별 변환 결과:")
sample_labels = np.arange(10, dtype=np.uint8)
sample_targets = T.to_regression(sample_labels)
for digit, target in zip(sample_labels, sample_targets):
    print(f"  label={digit}  →  {target[0]:.4f}  (= {digit}/9)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 변환 함수 그래프
digit_range = np.arange(10)
target_range = digit_range / 9.0
axes[0].plot(digit_range, target_range, 'o-', color='darkorange',
             linewidth=2, markersize=8)
axes[0].set_title("to_regression: label → target")
axes[0].set_xlabel("label (integer)")
axes[0].set_ylabel("target (label / 9.0)")
axes[0].set_xticks(range(10))
axes[0].set_ylim(-0.05, 1.1)
axes[0].grid(alpha=0.3)
for d, t in zip(digit_range, target_range):
    axes[0].annotate(f"{t:.3f}", (d, t), textcoords="offset points",
                     xytext=(0, 8), ha='center', fontsize=8)

# 실제 데이터 분포
axes[1].hist(labels_reg.flatten(), bins=20, color='darkorange',
             alpha=0.7, edgecolor='white')
axes[1].set_title("target distribution (train 60,000 samples)")
axes[1].set_xlabel("target value")
axes[1].set_ylabel("count")
axes[1].grid(axis='y', alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
assert labels_reg.shape == (60000, 1)
assert labels_reg.dtype == np.float32
assert labels_reg.min() >= 0.0 and labels_reg.max() <= 1.0

# 경계값 검증
label_zero_mask = labels_raw == 0
assert np.allclose(labels_reg[label_zero_mask], 0.0, atol=1e-6)

label_nine_mask = labels_raw == 9
assert np.allclose(labels_reg[label_nine_mask], 1.0, atol=1e-6)

# 전체 값 검증
expected = labels_raw.astype(np.float32).reshape(-1, 1) / 9.0
assert np.allclose(labels_reg, expected, atol=1e-6)

print("to_regression validation passed")

## 3. digit별 target 분포

train 데이터에서 각 digit이 얼마나 등장하는지, 해당 digit의 target 값은 무엇인지 확인한다.
10개 digit이 균일하게 분포하므로 각 target 값도 거의 동일한 빈도로 나타나야 한다.

In [ ]:
print("digit별 등장 횟수와 target 값:")
print(f"{'digit':>6}  {'target':>8}  {'count':>7}")
print("-" * 30)
for d in range(10):
    cnt = int((labels_raw == d).sum())
    tgt = d / 9.0
    print(f"  {d:4d}  {tgt:8.4f}  {cnt:7d}")
print("-" * 30)
print(f"  total           {len(labels_raw):7d}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

digit_counts = [(labels_raw == d).sum() for d in range(10)]
target_values = [d / 9.0 for d in range(10)]

cmap = plt.cm.get_cmap('plasma', 10)
bars = ax.bar(range(10), digit_counts, color=[cmap(i/9) for i in range(10)],
              alpha=0.85, edgecolor='white')

for bar, tgt in zip(bars, target_values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 50,
            f"t={tgt:.2f}", ha='center', va='bottom', fontsize=9)

ax.set_title("digit distribution with regression target values")
ax.set_xlabel("digit (label)")
ax.set_ylabel("count")
ax.set_xticks(range(10))
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
plt.show()

## 4. RegressionDataset 생성 및 확인

`RegressionDataset`은 인자 없이 `split`만 지정하면 `normalize + to_flat`과 `to_regression`을 기본값으로 적용한다.
`__getitem__`이 반환하는 `(image, target)` tuple에서 target은 `(1,)` shape 연속값이다.

In [ ]:
train_ds = RegressionDataset("train")
test_ds  = RegressionDataset("test")

print(f"train: len={len(train_ds)}")
print(f"  images.shape={train_ds.images.shape}, images.dtype={train_ds.images.dtype}")
print(f"  targets.shape={train_ds.targets.shape}, targets.dtype={train_ds.targets.dtype}")
print(f"  targets.min={train_ds.targets.min():.4f}, targets.max={train_ds.targets.max():.4f}")
print()
print(f"test: len={len(test_ds)}")
print(f"  images.shape={test_ds.images.shape}, images.dtype={test_ds.images.dtype}")
print(f"  targets.shape={test_ds.targets.shape}, targets.dtype={test_ds.targets.dtype}")

img, tgt = train_ds[0]
print(f"\ntrain_ds[0]: image.shape={img.shape}, target.shape={tgt.shape}, target={tgt}")

In [ ]:
assert len(train_ds) == 60000
assert len(test_ds) == 10000

assert train_ds.images.shape == (60000, 784)
assert train_ds.images.dtype == np.float32
assert train_ds.images.min() >= 0.0 and train_ds.images.max() <= 1.0

assert train_ds.targets.shape == (60000, 1)
assert train_ds.targets.dtype == np.float32
assert train_ds.targets.min() >= 0.0 and train_ds.targets.max() <= 1.0

img, tgt = train_ds[0]
assert img.shape == (784,)
assert tgt.shape == (1,)

print("RegressionDataset validation passed")

## 5. Dataloader 배치 생성

`Dataloader`는 `RegressionDataset`을 받아 `batch_size` 단위의 이미지 배치와 target 배치를 yield한다.
배치의 y shape이 `(batch_size, 1)`임을 확인한다. 이는 identity 활성화 출력 `(N, 1)`과 MSE 계산 시 shape을 맞추기 위해서이다.

In [ ]:
import math

BATCH_SIZE = 256
train_loader = Dataloader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = Dataloader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f"train_loader: len={len(train_loader)} batches")
print(f"test_loader : len={len(test_loader)} batches")

x_batch, y_batch = next(iter(train_loader))
print(f"\nfirst batch: x.shape={x_batch.shape}, y.shape={y_batch.shape}")
print(f"  x.dtype={x_batch.dtype}, y.dtype={y_batch.dtype}")
print(f"  y.min={y_batch.min():.4f}, y.max={y_batch.max():.4f}")
print(f"  y.mean={y_batch.mean():.4f}  (이상적 기댓값 ≈ 0.5)")

In [ ]:
# 학습 루프 시연 — 1 epoch, 첫 3 배치만 출력
print("학습 루프 시연 (1 epoch, 첫 3 배치):")
for batch_idx, (x, y) in enumerate(train_loader):
    if batch_idx < 3:
        print(f"  batch {batch_idx:3d}: x={x.shape}, y={y.shape}  "
              f"y.mean={y.mean():.4f}")
    elif batch_idx == 3:
        print("  ...")
        break

In [ ]:
assert len(train_loader) == math.ceil(len(train_ds) / BATCH_SIZE)
assert sum(len(x) for x, _ in train_loader) == len(train_ds)

x0, y0 = next(iter(train_loader))
assert x0.shape == (BATCH_SIZE, 784)
assert y0.shape == (BATCH_SIZE, 1)
assert x0.dtype == np.float32
assert y0.dtype == np.float32
assert y0.min() >= 0.0 and y0.max() <= 1.0, "target out of [0, 1] range"

print("Dataloader validation passed")

## 6. 샘플 이미지 시각화

train Dataset에서 digit 0~9를 하나씩 꺼내 이미지와 regression target 값을 함께 표시한다.
예측 결과를 복원할 때는 `round(target * 9)`를 적용하여 원래 digit을 얻는다.

In [ ]:
labels_raw = load_labels("train")

# digit 0~9에서 각 1개씩 대표 샘플 선택
representative_indices = [np.where(labels_raw == d)[0][0] for d in range(10)]

fig, axes = plt.subplots(2, 5, figsize=(14, 5))
cmap = plt.cm.get_cmap('plasma', 10)

for col, (digit, idx) in enumerate(zip(range(10), representative_indices)):
    row = col // 5
    c   = col % 5
    img, tgt = train_ds[idx]
    recovered = round(tgt[0] * 9)
    axes[row, c].imshow(img.reshape(28, 28), cmap=plt.cm.get_cmap('gray'))
    axes[row, c].set_title(
        f"digit={digit}\n"
        f"target={tgt[0]:.3f}\n"
        f"recover={recovered}",
        fontsize=9
    )
    axes[row, c].axis('off')

fig.suptitle("RegressionDataset — digit 0~9 representative samples", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# 복원 정확도 검증: target * 9 → round → clip → 원래 레이블
targets_all = train_ds.targets.flatten()             # (60000,)
recovered   = np.clip(np.round(targets_all * 9), 0, 9).astype(int)
labels_int  = labels_raw.astype(int)

accuracy = (recovered == labels_int).mean()
print(f"복원 정확도: {accuracy*100:.2f}%  (target * 9 → round → clip)")
assert accuracy == 1.0, f"복원 실패: {accuracy}"
print("regression target 복원 validation passed")

## 요약

이 노트북에서 확인한 내용은 다음과 같다.

| 항목 | 입력 | 출력 | 비고 |
|---|---|---|---|
| `to_regression` | `(N,) uint8` | `(N, 1) float32` | `/9.0` — 0→0.0, 9→1.0 |
| `RegressionDataset` | `split` | `images (N,784)`, `targets (N,1)` | eager 변환 |
| `Dataloader` | `RegressionDataset` | `(x, y)` batch tuple | y shape `(B, 1)` |
| 복원 | `target * 9` → `round` → `clip` | 원래 digit | 정확도 100% |

**핵심 설계 원칙**
- `to_regression`은 레이블을 `[0, 1]` 범위로 정규화하여 MSE loss 계산 시 예측값과 스케일을 맞춘다.
- target shape을 `(N, 1)`로 설정하여 identity 활성화 출력 `(N, 1)`과 shape이 일치하도록 한다.
- 예측값에 9를 곱하고 반올림·클리핑하면 원래 digit을 완전히 복원할 수 있다.